In [25]:
# Import necessary libraries

import pandas as pd # For data manipulation and analysis
import numpy as np # For numerical operations
import pickle # For serializing and deserializing Python objects
from sklearn.metrics import roc_auc_score
import plotly.graph_objects as go

import sys
sys.path.append("/home/suraj/Repositories/TumorImagingBench/notebooks/modelling")

from modelling_utils import (
    train_knn_classifier, evaluate_model,
    train_linear_probing_classifier,
    train_few_shot_classifier,
    build_knn_ensemble_classifier, predict_with_ensemble,
    train_stacking_ensemble_classifier, predict_with_stacking_ensemble,
    plot_model_comparison, extract_model_features, 
    compute_knn_indices, compute_overlap_matrix, plot_overlap_matrix, 
    split_shuffle_data
)


In [28]:
# Load features from a pickle file
feature_dict_path = "/home/suraj/Repositories/TumorImagingBench/data/features/dlcs.pkl" # Path to the pickle file containing features
with open(feature_dict_path, 'rb') as file: # Open the file in read binary mode
    data = pickle.load(file) # Load the data from the pickle file

In [29]:
# Store test accuracies for each model
test_accuracies_dict = {} # Initialize an empty dictionary to store test accuracies

# Iterate through each model's features
for model_name, values in data.items(): # Loop through each model and its corresponding features
    # Extract paths and labels for train, val, and test sets
    train_labels = [v["row"]["Malignant_lbl"] for v in values["train"]] # Extract malignancy labels for the training set
    val_labels = [v["row"]["Malignant_lbl"] for v in values["val"]] # Extract malignancy labels for the validation set
    
    # Stack features
    train_items = np.vstack([v["feature"] for v in values["train"]]) # Stack features for the training set
    val_items = np.vstack([v["feature"] for v in values["val"]]) # Stack features for the validation set
    
    # Concatenate train and val
    all_items = np.vstack((train_items, val_items))
    all_labels = train_labels + val_labels

  # Average across multiple shuffle splits
    n_splits = 10
    split_scores = []
    
    for split in range(n_splits):
        # Get stratified indices for new train/val split with different random seeds
        train_items, train_labels, val_items, val_labels, test_items, test_labels = split_shuffle_data(
            all_items, all_labels, train_ratio=0.5, val_ratio=0.2, random_seed=10+split, stratify=True
        )

        # Train model with hyperparameter optimization
        best_model, study = train_knn_classifier(train_items, train_labels, val_items, val_labels)
        
        # Get score for this split
        split_score = evaluate_model(best_model, test_items, test_labels)
        split_scores.append(split_score)
    
    # Average the scores across splits
    avg_score = np.mean(split_scores)
    
    # Compute the 95% confidence interval assuming normality
    std_error = np.std(split_scores, ddof=1) / np.sqrt(n_splits)
    margin = 1.96 * std_error
    ci_lower = avg_score - margin
    ci_upper = avg_score + margin
    
    # Evaluate on test set and store mean score with 95% CI
    test_accuracies_dict[model_name] = {"mean": avg_score, "ci95": (ci_lower, ci_upper)}

[I 2025-11-25 18:39:53,506] A new study created in memory with name: no-name-e339f61a-b732-4453-bac7-a5e977594ac8
[I 2025-11-25 18:39:53,523] Trial 0 finished with value: 0.49401669758812616 and parameters: {'k': 29}. Best is trial 0 with value: 0.49401669758812616.
[I 2025-11-25 18:39:53,535] Trial 1 finished with value: 0.48571428571428577 and parameters: {'k': 12}. Best is trial 0 with value: 0.49401669758812616.
[I 2025-11-25 18:39:53,547] Trial 2 finished with value: 0.47379406307977734 and parameters: {'k': 11}. Best is trial 0 with value: 0.49401669758812616.
[I 2025-11-25 18:39:53,561] Trial 3 finished with value: 0.46284786641929504 and parameters: {'k': 42}. Best is trial 0 with value: 0.49401669758812616.
[I 2025-11-25 18:39:53,573] Trial 4 finished with value: 0.4788497217068646 and parameters: {'k': 3}. Best is trial 0 with value: 0.49401669758812616.
[I 2025-11-25 18:39:53,586] Trial 5 finished with value: 0.48951762523191095 and parameters: {'k': 28}. Best is trial 0 wit

In [49]:
# Plot test accuracies
fig = plot_model_comparison(test_accuracies_dict, font_size=30, height=1200, width=800, marker_color="#FCA308")
fig.show() # Show the plot

In [50]:
test_accuracies_dict

{'CTClipVitExtractor': {'mean': 0.49447235154782315,
  'ci95': (0.46715392289644125, 0.5217907801992051)},
 'CTFMExtractor': {'mean': 0.5920342236379972,
  'ci95': (0.5766146762970503, 0.607453770978944)},
 'FMCIBExtractor': {'mean': 0.6757963734378828,
  'ci95': (0.6551453304354579, 0.6964474164403077)},
 'MerlinExtractor': {'mean': 0.5616576819407009,
  'ci95': (0.5414973807191465, 0.5818179831622552)},
 'ModelsGenExtractor': {'mean': 0.6453197745650576,
  'ci95': (0.62444825454842, 0.6661912945816952)},
 'PASTAExtractor': {'mean': 0.5563015600751451,
  'ci95': (0.5319988339945708, 0.5806042861557194)},
 'SUPREMExtractor': {'mean': 0.54421914563424,
  'ci95': (0.5271162865102295, 0.5613220047582506)},
 'VISTA3DExtractor': {'mean': 0.607937188597566,
  'ci95': (0.5892120340124767, 0.6266623431826552)},
 'VocoExtractor': {'mean': 0.541570693457486,
  'ci95': (0.5227371643281715, 0.5604042225868004)},
 'DummyResNetExtractor': {'mean': 0.5142019929755779,
  'ci95': (0.4884664912378015, 0

In [51]:
model_features = extract_model_features(data)
model_neighbors = compute_knn_indices(model_features, num_neighbors=10, metric="cosine")
overlap_matrix, model_list = compute_overlap_matrix(model_neighbors)
fig = plot_overlap_matrix(overlap_matrix, model_list, font_size=30, tickangle=45)
fig.show()

## Linear Probing Evaluation

Evaluate foundation model features using linear probing (logistic regression).
This complements KNN probing and is the standard transfer learning baseline.

In [47]:
# Linear Probing - Train logistic regression on frozen features
linear_probing_results = {}


for model_name, values in data.items():
    print(f"Linear Probing - {model_name}...")
    splits = [s for s in ["train", "val", "test"] if s in values and values[s]]
    if not splits:
        print("  Skipping: no splits found")
        continue

    labels = []
    features = []
    for split in splits:
        labels.extend([v["row"]["Malignant_lbl"] for v in values[split]])
        features.append(np.vstack([v["feature"] for v in values[split]]))

    labels_arr = np.array(labels)
    if labels_arr.dtype.kind in {"f", "c"}:
        mask = ~np.isnan(labels_arr)
    else:
        mask = np.ones_like(labels_arr, dtype=bool)

    all_items = np.vstack(features)[mask]
    all_labels = labels_arr[mask].tolist()

    n_splits = 10
    linear_split_scores = []

    for split in range(n_splits):
        train_items_s, train_labels_s, val_items_s, val_labels_s, test_items_s, test_labels_s = split_shuffle_data(
            all_items, all_labels, train_ratio=0.5, val_ratio=0.2, random_seed=10+split, stratify=True
        )

        linear_model, _ = train_linear_probing_classifier(train_items_s, train_labels_s, val_items_s, val_labels_s)
        linear_score = evaluate_model(linear_model, test_items_s, test_labels_s)
        linear_split_scores.append(linear_score)

    avg_score = np.mean(linear_split_scores)
    std_error = np.std(linear_split_scores, ddof=1) / np.sqrt(n_splits)
    margin = 1.96 * std_error
    ci_lower = avg_score - margin
    ci_upper = avg_score + margin

    linear_probing_results[model_name] = {"mean": avg_score, "ci95": (ci_lower, ci_upper)}
    print(f"  Linear Probing AUC: {avg_score:.4f} ± {margin:.4f}")

print("\n✓ Linear probing evaluation complete")


Linear Probing - CTClipVitExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.5457 ± 0.0331
Linear Probing - CTFMExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.5938 ± 0.0163
Linear Probing - FMCIBExtractor...


...

  Linear Probing AUC: 0.5985 ± 0.0196
Linear Probing - VocoExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.5637 ± 0.0159
Linear Probing - DummyResNetExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.5232 ± 0.0226

✓ Linear probing evaluation complete


In [52]:
linear_probing_results

{'CTClipVitExtractor': {'mean': 0.5456750796373437,
  'ci95': (0.5126207326174175, 0.57872942665727)},
 'CTFMExtractor': {'mean': 0.5937964551172098,
  'ci95': (0.5774652405323519, 0.6101276697020677)},
 'FMCIBExtractor': {'mean': 0.6760189496038553,
  'ci95': (0.6522260842779617, 0.699811814929749)},
 'MerlinExtractor': {'mean': 0.6415216858613084,
  'ci95': (0.6202818468256291, 0.6627615248969877)},
 'ModelsGenExtractor': {'mean': 0.6642571265212774,
  'ci95': (0.6492557631569993, 0.6792584898855556)},
 'PASTAExtractor': {'mean': 0.6058482398105041,
  'ci95': (0.5913977785710249, 0.6202987010499832)},
 'SUPREMExtractor': {'mean': 0.5784856652781182,
  'ci95': (0.542075258276991, 0.6148960722792454)},
 'VISTA3DExtractor': {'mean': 0.5985379400473739,
  'ci95': (0.578907844685245, 0.6181680354095028)},
 'VocoExtractor': {'mean': 0.5636731193334967,
  'ci95': (0.5478103483652362, 0.5795358903017572)},
 'DummyResNetExtractor': {'mean': 0.5232173486890468,
  'ci95': (0.500614636139842, 0.

## Few-Shot Learning Evaluation

Evaluate foundation model generalization with limited training data (1-shot, 5-shot, 10-shot).
This assesses how well models work in clinical settings with limited labels.

In [ ]:
# Few-Shot Learning - Evaluate with limited training samples
shot_configs = [1, 5, 10]
few_shot_results = {shots: {} for shots in shot_configs}

for model_name, values in data.items():
    print(f"Few-Shot Learning - {model_name}...")
    splits = [s for s in ["train", "val", "test"] if s in values and values[s]]
    if not splits:
        print("  Skipping: no splits found")
        continue


    labels = []
    features = []
    for split in splits:
        labels.extend([v["row"]["Malignant_lbl"] for v in values[split]])
        features.append(np.vstack([v["feature"] for v in values[split]]))

    labels_arr = np.array(labels)
    if labels_arr.dtype.kind in {"f", "c"}:
        mask = ~np.isnan(labels_arr)
    else:
        mask = np.ones_like(labels_arr, dtype=bool)

    all_items = np.vstack(features)[mask]
    all_labels = labels_arr[mask].tolist()

    for shots in shot_configs:
        n_splits = 10
        shot_scores = []

        for split in range(n_splits):
            train_items_s, train_labels_s, val_items_s, val_labels_s, test_items_s, test_labels_s = split_shuffle_data(
                all_items, all_labels, train_ratio=0.5, val_ratio=0.2, random_seed=10+split, stratify=True
            )

            np.random.seed(split)
            few_shot_model, _, _ = train_few_shot_classifier(
                train_items_s, train_labels_s,
                val_items_s, val_labels_s,
                shots=shots
            )

            test_score = evaluate_model(few_shot_model, test_items_s, test_labels_s)
            shot_scores.append(test_score)

        mean_score = np.mean(shot_scores)
        std_error = np.std(shot_scores, ddof=1) / np.sqrt(n_splits)
        margin = 1.96 * std_error

        few_shot_results[shots][model_name] = {"mean": mean_score, "ci95": (mean_score - margin, mean_score + margin)}

        if shots == 1:
            print(f"  {shots}-shot AUC: {mean_score:.4f} ± {margin:.4f} ... 10-shot: " , end="")
        elif shots == 10:
            print(f"{few_shot_results[shots][model_name]['mean']:.4f}")

print("\n✓ Few-shot learning evaluation complete")


## Few-Shot Learning Curves

Visualize how model performance scales with increasing training samples.

In [53]:
# Plot few-shot learning curves

model_names = list(next(iter(few_shot_results.values())).keys())
fig = go.Figure()

for model_name in model_names:
    shot_values = []
    means = []
    cis = []

    for shots in shot_configs:
        shot_values.append(shots)
        result = few_shot_results.get(shots, {}).get(model_name, {})
        if "mean" not in result or "ci95" not in result:
            means.append(float("nan"))
            cis.append(0)
        else:
            means.append(result["mean"])
            ci = result["ci95"][1] - result["mean"]
            cis.append(ci)

    fig.add_trace(go.Scatter(
        x=shot_values,
        y=means,
        error_y=dict(type='data', array=cis),
        mode='lines+markers',
        name=model_name,
        line=dict(width=2),
        marker=dict(size=8)
    ))

fig.update_layout(
    title='Few-Shot Learning Performance Scaling',
    xaxis_title='Number of Shots per Class',
    yaxis_title='Test AUC',
    height=600,
    width=1000,
    template='simple_white',
    hovermode='x unified'
)
fig.update_xaxes(tickvals=shot_configs)
fig.update_yaxes(range=[0, 1.0])

fig.show()

print("Key Insight: Steeper curves = better low-data generalization")
print("Models with 1-shot performance close to 10-shot are most data-efficient")


Key Insight: Steeper curves = better low-data generalization
Models with 1-shot performance close to 10-shot are most data-efficient


## Comparison: KNN vs Linear Probing vs Few-Shot

In [65]:
# Create comparison visualization
model_names = list(linear_probing_results.keys())

knn_means = [test_accuracies_dict[m]['mean'] for m in model_names]
linear_means = [linear_probing_results[m]['mean'] for m in model_names]
few_shot_10_means = [few_shot_results[10][m]['mean'] for m in model_names]

knn_errors = [test_accuracies_dict[m]['ci95'][1] - test_accuracies_dict[m]['mean'] for m in model_names]
linear_errors = [linear_probing_results[m]['ci95'][1] - linear_probing_results[m]['mean'] for m in model_names]
few_shot_errors = [few_shot_results[10][m]['ci95'][1] - few_shot_results[10][m]['mean'] for m in model_names]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=model_names,
    y=knn_means,
    error_y=dict(type='data', array=knn_errors),
    name='KNN Probing',
    marker_color='#4ECDC4'
))

fig.add_trace(go.Bar(
    x=model_names,
    y=linear_means,
    error_y=dict(type='data', array=linear_errors),
    name='Linear Probing',
    marker_color='#FF6B6B'
))

fig.add_trace(go.Bar(
    x=model_names,
    y=few_shot_10_means,
    error_y=dict(type='data', array=few_shot_errors),
    name='10-Shot Learning',
    marker_color='#95E1D3'
))

fig.update_layout(
    title='Evaluation Protocol Comparison',
    xaxis_title='Model',
    yaxis_title='Test AUC',
    barmode='group',
    height=600,
    width=900,
    template='simple_white',
    xaxis_tickangle=45
)
fig.update_yaxes(range=[0, 1.0])

fig.show()

print(f"Correlation KNN vs Linear Probing: {np.corrcoef(knn_means, linear_means)[0, 1]:.4f}")
print(f"Correlation KNN vs 10-Shot: {np.corrcoef(knn_means, few_shot_10_means)[0, 1]:.4f}")
print("Interpretation:")
print("  High KNN-Linear correlation (>0.8): Consistent feature quality assessment")
print("  10-Shot vs KNN: Difference shows data-efficiency gains with simple learning")


Correlation KNN vs Linear Probing: 0.8717
Correlation KNN vs 10-Shot: 0.8274
Interpretation:
  High KNN-Linear correlation (>0.8): Consistent feature quality assessment
  10-Shot vs KNN: Difference shows data-efficiency gains with simple learning


## Alignment-Based Ensemble Method

Combine all models using mutual k-NN overlap alignment as weights.
Models with high alignment with others are weighted more heavily.

In [66]:
# Build alignment-based ensemble
print("Building alignment-based ensemble...")

ensemble_features_dict = {}

first_model = list(data.keys())[0]
available_splits = [s for s in ["train", "val", "test"] if s in data[first_model] and data[first_model][s]]
if not available_splits:
    raise ValueError("No splits found for ensemble construction.")


labels = []
for split in available_splits:
    labels.extend([v["row"]["Malignant_lbl"] for v in data[first_model][split]])

labels_arr = np.array(labels)
if labels_arr.dtype.kind in {"f", "c"}:
    valid_mask = ~np.isnan(labels_arr)
else:
    valid_mask = np.ones_like(labels_arr, dtype=bool)
labels_arr = labels_arr[valid_mask]

for model_name, values in data.items():
    feat_blocks = []
    for split in available_splits:
        if split in values and values[split]:
            feat_blocks.append(np.vstack([v['feature'] for v in values[split]]))
    if feat_blocks:
        stacked = np.vstack(feat_blocks)[valid_mask]
    else:
        stacked = np.array([])
    ensemble_features_dict[model_name] = stacked

all_labels_ensemble = labels_arr.tolist()

n_splits = 10
ensemble_scores = []
individual_ensemble_scores = {m: [] for m in model_list}

for split_idx in range(n_splits):
    (train_idx, train_labels_s, val_idx, val_labels_s, 
     test_idx, test_labels_s) = split_shuffle_data(
        np.arange(len(all_labels_ensemble)), all_labels_ensemble,
        train_ratio=0.5, val_ratio=0.2, random_seed=10+split_idx, stratify=True
    )

    train_features_dict = {m: ensemble_features_dict[m][train_idx] for m in model_list}
    val_features_dict = {m: ensemble_features_dict[m][val_idx] for m in model_list}
    test_features_dict = {m: ensemble_features_dict[m][test_idx] for m in model_list}

    ensemble_model, _ = build_knn_ensemble_classifier(
        train_features_dict, train_labels_s,
        val_features_dict, val_labels_s,
        overlap_matrix.copy(), model_list, k=10
    )

    ensemble_test_preds = predict_with_ensemble(ensemble_model, test_features_dict, model_list)

    if ensemble_test_preds.shape[1] == 2:
        ensemble_auc = roc_auc_score(test_labels_s, ensemble_test_preds[:, 1])
    else:
        ensemble_auc = roc_auc_score(test_labels_s, ensemble_test_preds, multi_class='ovr')

    ensemble_scores.append(ensemble_auc)

    from sklearn.neighbors import KNeighborsClassifier
    for model_name in model_list:
        knn = KNeighborsClassifier(n_neighbors=10, metric='cosine')
        knn.fit(train_features_dict[model_name], train_labels_s)
        test_preds = knn.predict_proba(test_features_dict[model_name])

        if test_preds.shape[1] == 2:
            model_auc = roc_auc_score(test_labels_s, test_preds[:, 1])
        else:
            model_auc = roc_auc_score(test_labels_s, test_preds, multi_class='ovr')

        individual_ensemble_scores[model_name].append(model_auc)

    if (split_idx + 1) % 5 == 0:
        print(f"  Completed {split_idx + 1}/{n_splits} splits")

ensemble_mean = np.mean(ensemble_scores)
ensemble_std = np.std(ensemble_scores, ddof=1) / np.sqrt(n_splits)
ensemble_ci = 1.96 * ensemble_std

print(f"✓ Ensemble evaluation complete")
print(f"Ensemble Performance:")
print(f"  Test AUC: {ensemble_mean:.4f} ± {ensemble_ci:.4f}")

print(f"Comparison to Individual Models:")
best_model_name = None
best_model_score = 0
for model_name in model_list:
    ind_mean = np.mean(individual_ensemble_scores[model_name])
    ind_std = np.std(individual_ensemble_scores[model_name], ddof=1) / np.sqrt(n_splits)
    ind_ci = 1.96 * ind_std
    improvement = ensemble_mean - ind_mean

    if ind_mean > best_model_score:
        best_model_score = ind_mean
        best_model_name = model_name

    print(f"  {model_name}: {ind_mean:.4f} ± {ind_ci:.4f}  (ensemble: {improvement:+.4f})")

print(f"Best Single Model: {best_model_name} ({best_model_score:.4f})")
print(f"Ensemble Advantage: {ensemble_mean - best_model_score:+.4f}")


Building alignment-based ensemble...
  Completed 5/10 splits
  Completed 10/10 splits
✓ Ensemble evaluation complete
Ensemble Performance:
  Test AUC: 0.6585 ± 0.0130
Comparison to Individual Models:
  CTClipVitExtractor: 0.5003 ± 0.0160  (ensemble: +0.1582)
  CTFMExtractor: 0.5889 ± 0.0157  (ensemble: +0.0696)
  FMCIBExtractor: 0.6674 ± 0.0150  (ensemble: -0.0088)
  MerlinExtractor: 0.5642 ± 0.0131  (ensemble: +0.0943)
  ModelsGenExtractor: 0.6205 ± 0.0087  (ensemble: +0.0380)
  PASTAExtractor: 0.5568 ± 0.0223  (ensemble: +0.1018)
  SUPREMExtractor: 0.5236 ± 0.0169  (ensemble: +0.1349)
  VISTA3DExtractor: 0.5926 ± 0.0211  (ensemble: +0.0659)
  VocoExtractor: 0.5223 ± 0.0200  (ensemble: +0.1362)
  DummyResNetExtractor: 0.5231 ± 0.0331  (ensemble: +0.1354)
Best Single Model: FMCIBExtractor (0.6674)
Ensemble Advantage: -0.0088


## Ensemble vs Single Models

Bar plot comparing the alignment-weighted ensemble to each individual model.

In [68]:
# Plot ensemble vs single-model performance
model_names_plot = list(individual_ensemble_scores.keys())
ind_means = [np.mean(individual_ensemble_scores[m]) for m in model_names_plot]
ind_errors = [1.96 * np.std(individual_ensemble_scores[m], ddof=1) / np.sqrt(len(individual_ensemble_scores[m])) for m in model_names_plot]

bar_names = model_names_plot + ["Ensemble"]
bar_means = ind_means + [ensemble_mean]
bar_errors = ind_errors + [ensemble_ci]

colors = ['#4ECDC4'] * len(model_names_plot) + ['#FCA308']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=bar_names,
    y=bar_means,
    error_y=dict(type='data', array=bar_errors),
    marker_color=colors,
    text=[f"{m:.3f}" for m in bar_means],
    textposition='auto'
))

fig.update_layout(
    title='Ensemble vs Individual Models',
    xaxis_title='Model',
    yaxis_title='Test AUC',
    height=600,
    width=1200,
    template='simple_white',
    xaxis_tickangle=45
)
fig.update_yaxes(range=[0, 1.0])

fig.show()

best_ind_mean = max(ind_means) if ind_means else float('nan')
print(f"Ensemble uplift over best single: {ensemble_mean - best_ind_mean:+.4f}")


Ensemble uplift over best single: -0.0088


## Ensemble Model Weights

Visualize the alignment-based weights assigned to each model.

In [69]:
# Display ensemble weights from the first evaluation
train_idx, train_labels_s, val_idx, val_labels_s, test_idx, test_labels_s = split_shuffle_data(
    np.arange(len(all_labels_ensemble)), all_labels_ensemble,
    train_ratio=0.5, val_ratio=0.2, random_seed=50, stratify=True
)

train_features_dict = {m: ensemble_features_dict[m][train_idx] for m in model_list}
val_features_dict = {m: ensemble_features_dict[m][val_idx] for m in model_list}

final_ensemble, _ = build_knn_ensemble_classifier(
    train_features_dict, train_labels_s,
    val_features_dict, val_labels_s,
    overlap_matrix.copy(), model_list, k=10
)

weights = final_ensemble['weights']
models_for_plot = list(weights.keys())
weight_values = list(weights.values())

fig = go.Figure()
fig.add_trace(go.Bar(
    x=models_for_plot,
    y=weight_values,
    marker_color='#FCA308',
    text=[f'{w:.3f}' for w in weight_values],
    textposition='auto',
))

fig.update_layout(
    title='Ensemble Model Weights (Based on k-NN Alignment)',
    xaxis_title='Model',
    yaxis_title='Weight',
    height=500,
    width=700,
    template='simple_white',
    xaxis_tickangle=45
)
fig.update_yaxes(range=[0, max(weight_values) * 1.15])

fig.show()

print(f"Weight Statistics:")
print(f"  Max weight: {max(weight_values):.4f}")
print(f"  Min weight: {min(weight_values):.4f}")
print(f"  All weights sum to: {sum(weight_values):.4f}")
print("Interpretation:")
print("  Higher weight = better alignment with other models (consensus)")
print("  Lower weight = unique/diverse representation (adds complementary info)")


Weight Statistics:
  Max weight: 0.1604
  Min weight: 0.0163
  All weights sum to: 1.0000
Interpretation:
  Higher weight = better alignment with other models (consensus)
  Lower weight = unique/diverse representation (adds complementary info)


## Stacked Ensemble with Learned Weights

Train a logistic-regression meta-learner on top of per-model k-NN probabilities.

In [70]:
# Stacking ensemble with learned weights (meta-learned combination of models)
from sklearn.neighbors import KNeighborsClassifier

print("Evaluating stacking ensemble with learned weights...")

n_splits = 10
stacking_scores = []
stacking_val_scores = []
last_stacking_model = None
stacking_base_scores = {m: [] for m in model_list}

for split_idx in range(n_splits):
    (train_idx, train_labels_s, val_idx, val_labels_s,
     test_idx, test_labels_s) = split_shuffle_data(
        np.arange(len(all_labels_ensemble)), all_labels_ensemble,
        train_ratio=0.5, val_ratio=0.2, random_seed=110 + split_idx, stratify=True
    )

    train_features_dict = {m: ensemble_features_dict[m][train_idx] for m in model_list}
    val_features_dict = {m: ensemble_features_dict[m][val_idx] for m in model_list}
    test_features_dict = {m: ensemble_features_dict[m][test_idx] for m in model_list}

    stacking_model, val_auc = train_stacking_ensemble_classifier(
        train_features_dict, train_labels_s,
        val_features_dict, val_labels_s,
        k_candidates=(5, 10, 15, 25),
        meta_C_candidates=(0.25, 1.0, 4.0),
    )
    last_stacking_model = stacking_model
    stacking_val_scores.append(val_auc)

    test_pred = predict_with_stacking_ensemble(stacking_model, test_features_dict)
    if test_pred.shape[1] == 2:
        test_auc = roc_auc_score(test_labels_s, test_pred[:, 1])
    else:
        test_auc = roc_auc_score(test_labels_s, test_pred, multi_class='ovr')
    stacking_scores.append(test_auc)

    k_for_base = stacking_model['k']
    for model_name in model_list:
        knn = KNeighborsClassifier(n_neighbors=k_for_base, metric="cosine")
        knn.fit(train_features_dict[model_name], train_labels_s)
        base_pred = knn.predict_proba(test_features_dict[model_name])
        if base_pred.shape[1] == 2:
            base_auc = roc_auc_score(test_labels_s, base_pred[:, 1])
        else:
            base_auc = roc_auc_score(test_labels_s, base_pred, multi_class='ovr')
        stacking_base_scores[model_name].append(base_auc)

    if (split_idx + 1) % 5 == 0:
        print(f"  Completed {split_idx + 1}/{n_splits} splits")

stacking_mean = np.mean(stacking_scores)
stacking_ci = 1.96 * np.std(stacking_scores, ddof=1) / np.sqrt(n_splits)
val_mean = np.mean(stacking_val_scores)

print(f"Stacking ensemble test AUC: {stacking_mean:.4f} ± {stacking_ci:.4f}")
print(f"Validation AUC (meta search average): {val_mean:.4f}")

best_single = None
best_single_score = -np.inf
for model_name, scores in stacking_base_scores.items():
    mean_score = np.mean(scores)
    if mean_score > best_single_score:
        best_single_score = mean_score
        best_single = model_name

print(f"Best single model (matched k): {best_single} — {best_single_score:.4f}")
print(f"Ensemble advantage over best single: {stacking_mean - best_single_score:+.4f}")


Evaluating stacking ensemble with learned weights...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-package

  Completed 5/10 splits


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-package

  Completed 10/10 splits
Stacking ensemble test AUC: 0.6723 ± 0.0247
Validation AUC (meta search average): 0.7806
Best single model (matched k): FMCIBExtractor — 0.6719
Ensemble advantage over best single: +0.0004


In [71]:
# Visualize meta-learner weights from the last stacking run
if last_stacking_model is None:
    raise RuntimeError("Run the stacking ensemble cell before visualizing weights.")

meta_model = last_stacking_model['meta_model']
model_list = last_stacking_model['model_list']
coef = meta_model.coef_.mean(axis=0)

n_models = len(model_list)
cols_per_model = coef.shape[0] // n_models if n_models else 0

weight_rows = []
for idx, name in enumerate(model_list):
    start = idx * cols_per_model
    end = start + cols_per_model
    block = coef[start:end]
    weight_rows.append({
        "model": name,
        "meta_weight": float(np.mean(block))
    })

weight_df = pd.DataFrame(weight_rows)
weight_df['normalized'] = np.exp(weight_df['meta_weight']) / np.exp(weight_df['meta_weight']).sum()
weight_df = weight_df.sort_values('normalized', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=weight_df['model'],
    y=weight_df['normalized'],
    marker_color='#4ECDC4',
    text=[f"{w:.3f}" for w in weight_df['normalized']],
    textposition='auto'
))
fig.update_layout(
    title='Stacking Ensemble Meta-weights (softmax-normalized coefficients)',
    xaxis_title='Model',
    yaxis_title='Normalized weight',
    height=500,
    width=700,
    template='simple_white',
    xaxis_tickangle=45,
)
fig.update_yaxes(range=[0, weight_df['normalized'].max() * 1.15])

fig.show()

print("Raw meta coefficients (per-model mean):")
print(weight_df[['model', 'meta_weight']].to_string(index=False))


Raw meta coefficients (per-model mean):
               model  meta_weight
  ModelsGenExtractor     3.072156
      FMCIBExtractor     2.791631
       CTFMExtractor     1.968770
      PASTAExtractor     0.825271
    VISTA3DExtractor     0.593726
     SUPREMExtractor     0.548908
       VocoExtractor     0.465957
     MerlinExtractor     0.268804
  CTClipVitExtractor    -2.613571
DummyResNetExtractor    -3.152090


## Ensemble Comparison Summary

Visualize alignment ensemble, stacked ensemble, and the best single model in one chart.

In [75]:
# Compare ensembles against best single model
if 'ensemble_mean' not in globals() or 'stacking_mean' not in globals():
    raise RuntimeError("Run alignment and stacking sections first.")

best_single_mean = best_model_score
best_single_name = best_model_name
best_single_ci = 1.96 * np.std(individual_ensemble_scores[best_single_name], ddof=1) / np.sqrt(len(individual_ensemble_scores[best_single_name]))

labels = [f"Best single ({best_single_name})", "Alignment ensemble", "Stacking ensemble"]
means = [best_single_mean, ensemble_mean, stacking_mean]
errors = [best_single_ci, ensemble_ci, stacking_ci]
colors = ['#4ECDC4', '#FCA308', '#FF6B6B']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels,
    y=means,
    error_y=dict(type='data', array=errors),
    marker_color=colors,
    text=[f"{m:.3f}" for m in means],
    textposition='auto'
))

fig.update_layout(
    title='Alignment vs Stacking vs Best Single',
    xaxis_title='Model',
    yaxis_title='Test AUC',
    height=500,
    width=600,
    template='simple_white',
    xaxis_tickangle=20
)
fig.update_yaxes(range=[0, 1.0])
fig.show()

print(f"Stacking uplift over best single: {stacking_mean - best_single_mean:+.4f}")
print(f"Stacking uplift over alignment ensemble: {stacking_mean - ensemble_mean:+.4f}")


Stacking uplift over best single: +0.0050
Stacking uplift over alignment ensemble: +0.0138
